In [ ]:
! pip install matplotlib seaborn scikit-learn flask mlflow

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
mlflow.set_experiment("Aula_4")

/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2025/11/13 19:48:26 INFO mlflow.tracking.fluent: Experiment with name 'Aula_4' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///home/teru/aula_monitoramento/mlruns/117064650587655535', creation_time=1763074106358, experiment_id='117064650587655535', last_update_time=1763074106358, lifecycle_stage='active', name='Aula_4', tags={}>

In [3]:
# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

In [ ]:
# Start an MLflow run
with mlflow.start_run() as run:
    # Log the hyperparameters
    mlflow.log_params(params)

    # Train the model
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)

    # Log the model
    mlflow.sklearn.log_model(sk_model=lr, name="iris_model", input_example=np.array([[5.1, 3.5, 1.4, 0.2]]))

    # Predict on the test set, compute and log the loss metric
    y_pred = lr.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Optional: Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")
    
    model_uri = f"runs:/{run.info.run_id}/iris_model"
    mv = mlflow.register_model(
        model_uri, "iris_model", tags={"version": "latest"}, artifact='name_labels.pkl'
    )

/home/teru/aula_monitoramento/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
Successfully registered model 'iris_model'.
2025/11/13 20:02:13 WARNING mlflow.tracking._model_registry.fluent: Run with id bbab922af92c4577bb727bf7309d1467 has no artifacts at artifact path 'iris_model', registering model based on models:/m-d6c97e6376d9484195ac03f40f520b1f instead
Created version '1' of model 'iris_model'.


[bash]

mlflow ui -p 5000

---

In [9]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle

import requests
import json

In [17]:
# 🔹 URL da API Flask (ajuste se necessário)
url = "http://localhost:5002/invocations"

with open("name_labels.pkl", "rb") as arquivo:
    target_names = pickle.load(arquivo)
    mlflow.log_artifact("name_labels.pkl")

In [15]:
payload = {"inputs": [5.1, 3.5, 1.4, 0.2]}  # setosa

pred = requests.post(url, json=payload).json()
print(pred)
pred = pred['predictions']['predicted_class']
predicted_name = target_names[pred[0]]

print(predicted_name)

{'erro': "string indices must be integers, not 'str'"}


KeyError: 'predictions'

In [ ]:
payload = {"inputs": [[6.2, 3.4, 5.4, 2.3]]}   # virginica

pred = requests.post(url, json=payload).json()
pred = pred['predictions']
predicted_name = target_names[pred[0]]

print(predicted_name)